# 04 — Synthetic Control Case Study

Builds a synthetic counterfactual for one treated state (a weighted blend of untreated donor states matched on pre-treatment characteristics), in the tradition of Abadie, Diamond & Hainmueller (2010) and Card & Krueger's NJ/PA study.

**Requires R** with the `Synth` package (`Rscript install.R`).

In [1]:
import sys
sys.path.insert(0, r"C:\Users\thoma\minimum-wage-causal-inference")


In [2]:
import shutil
HAS_R = shutil.which('Rscript') is not None
print(f'Rscript available: {HAS_R}')

Rscript available: False


In [3]:
from src.data.loader import load_state_year_panel
panel, is_synthetic = load_state_year_panel()

treated_candidates = (
    panel.loc[panel.get('treated', False) == True, 'state'].unique()
    if 'treated' in panel.columns else panel['state'].unique()
)
treated_state = sorted(treated_candidates)[0]
treatment_year = int(panel[panel.state == treated_state].adoption_year.iloc[0]) \
    if 'adoption_year' in panel.columns else int(panel.year.median())
print(f'Case study: {treated_state}, treatment_year={treatment_year}')

Case study: S02, treatment_year=2016


In [4]:
if HAS_R:
    from src.methods.r_bridge import run_synthetic_control
    sc_result = run_synthetic_control(panel, treated_state, treatment_year)
    display(sc_result)

    import matplotlib.pyplot as plt
    for series_type, group in sc_result.groupby('type'):
        plt.plot(group.year, group.unemployment_rate, label=series_type)
    plt.axvline(treatment_year, color='gray', linestyle='--')
    plt.legend(); plt.title(f'{treated_state}: actual vs. synthetic'); plt.show()
else:
    print('Skipping: install R + the Synth package to run this cell '
          '(see src/methods/synthetic_control.R).')

Skipping: install R + the Synth package to run this cell (see src/methods/synthetic_control.R).
